# Introduction to Profiling — Intel VTune
## Concepts and practice 

**This notebook runs inside the JupyterLab session you submit through ARE**, NCI's customised Open OnDemand service. Select **12 CPU cores on one node** in the ARE launch form. The running session supplies the resources for timing and profiling the **prebuilt LULESH application provided through the private `lulesh` module for project `vp91`**.



**By the end you should be able to:**

- Use the prebuilt, optimised LULESH executable and its matching source information.
- Establish a baseline and collect Hotspots data.
- Interpret sampled CPU time, elapsed time, and runtime waiting.
- Compare one MPI rank using **1, 2, 4, 6, and 12 OpenMP threads**.
- Compare eight MPI ranks with eight OpenMP threads on the same global mesh.
- Explain software and hardware sampling, and choose evidence to test a performance hypothesis.

Allow **5–10 minutes to check the session and module**, then **about one hour for the lesson**.  Keep this notebook beside `lab.py` in the repository's `vtune/` directory.

In [ ]:
import os
# the jupyter notebook is launched from your $HOME, change the working directory provided a username directory is created under /scratch/vp91
os.chdir(os.path.expandvars("/scratch/vp91/$USER/intro-to-profiling"))


## Exercise roadmap

| Stop | Time in core lesson | Edit / run | Check before continuing |
|---|---:|---|---|
| [ARE setup](#setup) | Before the lesson | Launch form, module, paths and environment | One running session with 12 cores on one node |
| [1. Establish a baseline](#exercise-1) | 10 min | One-thread warm-up and three repetitions | Successful, repeatable LULESH runs |
| [2. Find a hotspot](#exercise-2) | 10 min | Hotspots collection and reports | A function, its cost, and a saved result |
| [3. Interpret the evidence](#exercise-3) |  min | Timing example, reports, source | Explain the metric and its denominator |
| [4. Change one variable](#exercise-4) | 15 min | 1, 2, 4, 6, then 12 threads | Repeated timings and a supported conclusion |
| [5. Compare MPI and OpenMP](#exercise-5) | 15 min | Eight ranks versus eight threads; repeated timings and software Hotspots | Same global mesh, eight application cores, comparable numerical results |

Run the cells **in order and one experiment at a time**. The kernel launches the commands in the existing ARE job. Keep other notebooks and terminals idle during measurements: Jupyter, VTune, and LULESH share the session's resources. There are no precomputed performance results; fill in the worksheet from your own runs.

<a id="foundations"></a>
## 1. What does a profiler tell us?

A **timer** measures a defined interval. A **profiler** attributes cost to functions, source locations, threads, or hardware events. A hotspot is a place where substantial measured cost accumulates; it is a starting point for investigation.

### Sampling: take snapshots

For example, a profiler might check every **10 milliseconds** and record the executing function, instruction, and **call stack**. If **600 of 1,000 samples** land inside `calculate_forces()`, that suggests roughly **60% of the sampled CPU time** was spent there. This is a statistical estimate, not a count of function calls. Short runs can have too few samples to support a stable ranking.

### Instrumentation: record selected events

Instrumentation adds measurements at points such as function entry and exit. If **100 sequential calls** to `calculate_forces()` accumulate **2 seconds** of elapsed time, the average is **20 milliseconds per call**. Timestamp-based measurements describe elapsed time. Executing the measurement code adds overhead; sampling also has a cost.

```text
Correct result → repeatable baseline → collect → locate expensive code
      ↑                                             |
      └──── validate and re-time ← change one thing ← hypothesis
```

**Predict:** does the most frequently called function necessarily consume the most CPU time?

<details><summary>Check your reasoning</summary>

No. A million calls costing 100 nanoseconds of CPU time each total 0.1 seconds; ten calls costing 100 milliseconds each total 1 second. Measure an unprofiled baseline, then use the profile to investigate where the cost arises.

</details>

### Software sampling and hardware sampling

Both methods sample execution rather than record every function call. They differ in **what triggers a sample** and the evidence they can provide.

| | Software sampling (user-mode) | Hardware event-based sampling |
|---|---|---|
| Trigger | A periodic timer samples executing application threads. | A processor performance counter reaches a configured event threshold. |
| Evidence | Estimated CPU time in functions and call stacks. | Code associated with selected events, such as CPU cycles or completed instructions. Other analyses select events for cache and memory behaviour. |
| Requirements | Permission to profile the application; no hardware counters for our configuration. | A supported processor/VTune combination and permission to access the required counters through the OS or a sampling driver. |
| Workshop use | **Every profiling exercise.** | Explanations and reference examples only. |

**Illustrative software example:** sample every 10 ms of thread execution to estimate which functions occupy CPU time. The sample interval and profiler overhead affect the estimate.

**Illustrative hardware example:** sample after each million occurrences of a selected CPU-cycle event. This associates cycle activity with code; it does not record every cycle or function call. This threshold is an illustration, not a setting used by this notebook.

The CPU's **performance monitoring unit (PMU)** contains hardware counters. Counting totals and collecting event samples are different operations; access to one does not guarantee access to every analysis. Hardware sampling is also statistical, and ordinary software Hotspots cannot measure cache misses or memory bandwidth directly.

**In this notebook:** `lab.run(..., analysis="hotspots")` always selects `sampling-mode=sw` and disables additional hardware insights. Baseline and scaling timings run without a profiler.

Sources: [Intel user-mode sampling](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-2/user-mode-sampling-and-tracing-collection.html), [hardware event-based sampling](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-2/hw-event-based-sampling-collection.html), and [Hotspots modes](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-2/basic-hotspots-analysis.html).


### How the notebook uses the allocation

```text
Your browser → ARE submits one 12-core, single-node job
                   └─ JupyterLab and this Python kernel
                        └─ mpirun → [VTune →] prebuilt LULESH
                             └─ results in your own scratch directory
```

| Experiment | MPI ranks | OpenMP threads per rank | Application core budget |
|---|---:|---:|---:|
| Baseline / first Hotspots | 1 | 1 | 1 |
| Thread comparison | 1 | 2, 4, 6, 12 | 2, 4, 6, 12 |
| Second software Hotspots | 1 | 12 | 12 |
| Exercise 5: MPI | 8 | 1 | 8 |
| Exercise 5: OpenMP, same global mesh | 1 | 8 | 8 |

The **allocation remains 12 cores throughout**. A one-thread baseline intentionally uses a subset. Twelve allocated cores do not imply twelve MPI ranks: LULESH requires cubic rank counts, so its twelve-core run here uses **one rank × twelve threads**. 

### Locate the training files and module executable


```bash
cd /scratch/vp91/$USER
git clone https://github.com/LLNL/LULESH.git
```

**RUN the next cell.** It finds `lulesh2.0` on the `PATH` set by `module load lulesh`. 

Results go to **`/scratch/vp91/<your-user>/vtune-results/`**. The shared executable/source installation is used for reading and execution; it is not a results directory. 

In [ ]:
from pathlib import Path
import getpass
import json
import os
import shlex
import shutil
import socket
import statistics
import subprocess
import sys

PROJECT = "vp91"
WORK = Path(f"/scratch/{PROJECT}/{getpass.getuser()}")
candidates = [Path.cwd(), *Path.cwd().parents, WORK / "intro-to-profiling"]
REPO = next((p for p in candidates if (p / "vtune/lab.py").is_file()), None)
# If needed: REPO = Path("/scratch/vp91/your_user/intro-to-profiling")
if REPO is None:
    raise FileNotFoundError("Set REPO to the training checkout containing vtune/lab.py")
REPO = REPO.expanduser().resolve()

executable = shutil.which("lulesh2.0")
if executable is None:
    raise RuntimeError("lulesh2.0 is missing from the kernel PATH. Add lulesh to the ARE "
                       "Modules field, check its private module directory, and relaunch.")
LULESH_EXE = Path(executable).resolve()
source_setting = os.environ.get("LULESH_SOURCE")
LULESH_SOURCE = Path(source_setting).expanduser().resolve() if source_setting else None
# Optional: LULESH_SOURCE = Path("/matching/source/path/from/instructor")
RESULTS = WORK / "vtune-results"
ALLOCATED_CORES = 12
THREAD_COUNTS = [1, 2, 4, 6, 12]
PROFILE_THREADS = 12
sys.path.insert(0, str(REPO / "vtune"))
from lab import Lab, report, result_dirs

lab = Lab(binary=LULESH_EXE, results=RESULTS, allocated_cores=ALLOCATED_CORES,
          ranks=1, size=30, iterations=200)
print("Training repository:", REPO)
print("Module executable:", LULESH_EXE)
print("Matching source:", LULESH_SOURCE or "not supplied by the module")
print("Run budget:", ALLOCATED_CORES, "cores on one node")
print("Your results:", RESULTS)

In [ ]:
# Inspect this running session; this cell does not submit a job.
print("Host:", socket.gethostname())
print("PBS job:", os.environ.get("PBS_JOBID", "not exposed to the kernel"))
print("PBS_NCPUS:", os.environ.get("PBS_NCPUS", "not exposed to the kernel"))

pbs_cores = os.environ.get("PBS_NCPUS")
if pbs_cores is not None:
    if not pbs_cores.isdigit() or int(pbs_cores) != ALLOCATED_CORES:
        raise RuntimeError("This lesson expects the ARE session launched with 12 CPU cores")
else:
    print("Use the ARE session card to verify 12 cores; the lab budget remains 12.")

nodefile_name = os.environ.get("PBS_NODEFILE")
if nodefile_name:
    nodefile = Path(nodefile_name)
    if nodefile.is_file():
        hosts = {line.strip() for line in nodefile.read_text().splitlines() if line.strip()}
        print("Hosts in the allocation:", sorted(hosts))
        if len(hosts) != 1:
            raise RuntimeError("This workshop uses one node; check the ARE session settings")
    else:
        print("PBS_NODEFILE is not readable here; check the ARE session card for one node.")
else:
    print("Node list not exposed; check the ARE session card for one node.")

if hasattr(os, "sched_getaffinity"):
    visible_cpus = len(os.sched_getaffinity(0))
    print("Logical CPUs in this kernel's affinity mask:", visible_cpus)
    if visible_cpus < ALLOCATED_CORES:
        raise RuntimeError("Kernel affinity is too narrow for 12 threads; ask the instructor to check the session")
print("All experiments are capped at 12 cores, even if the host exposes more CPUs.")

In [ ]:
# Check runtime tools inherited from the ARE Modules setting.
required = ("lulesh2.0", "mpirun", "vtune")
missing = [tool for tool in required if not shutil.which(tool)]
if missing:
    raise RuntimeError("Missing tools: " + ", ".join(missing) +
                       ". Check lulesh/VTune and their runtime dependencies in ARE.")
mpi_version = subprocess.check_output(["mpirun", "--version"], text=True)
print(mpi_version)
if "Open MPI" not in mpi_version and "OpenRTE" not in mpi_version:
    raise RuntimeError("The private lulesh module must supply the matching Open MPI runtime")
vtune_version = subprocess.check_output(["vtune", "-version"], text=True)
print(vtune_version)
lab.validate(threads=PROFILE_THREADS)
if LULESH_SOURCE is None or not LULESH_SOURCE.is_dir():
    print("Matching source is unavailable. Profiling can proceed; obtain the source path "
          "from the instructor for Exercise 3 and set LULESH_SOURCE.")
lab.results.mkdir(parents=True, exist_ok=True)
(lab.results / "environment.txt").write_text(
    mpi_version + "\n" + vtune_version +
    "\nModules: " + os.environ.get("LOADEDMODULES", "unknown") +
    f"\nExecutable: {LULESH_EXE}\nSource: {LULESH_SOURCE}\n" +
    f"ARE allocation: {ALLOCATED_CORES} cores on one node\n")

### Why the supplied build is suitable for profiling

This training prepared the `lulesh` module with **MPI and OpenMP enabled** and an **optimised executable with debug symbols**. Participants use the same supplied binary for all comparisons.

| Build setting | Why it matters when reading a profile |
|---|---|
| `-O3` | Represents optimised application execution |
| `-g` | Retains symbols and source locations |
| `-fno-omit-frame-pointer` | Helps stack unwinding |
| MPI and OpenMP enabled | Supports the rank and thread experiments |



**There is no build step in the participant workflow.** Optional preparation instructions are in the [application setup notes](../docs/source/tutorial/intro.rst).

### LULESH: what is the workload?

LULESH is a C++ hydrodynamics proxy application. MPI partitions its three-dimensional mesh; OpenMP threads work within each rank.

- `-s N` sets the mesh side length **per rank**: total elements are `ranks × N³`.
- `-i K` limits the number of iterations; the simulation may reach its stop time sooner. Record the **actual** iteration count printed at completion.
- The MPI decomposition requires **1, 8, 27, … ranks**, a perfect cube. Four ranks are invalid for this application.

Start with **one rank** and vary threads. That holds the mesh and decomposition fixed and makes an introductory comparison easier to interpret. With Open MPI, `-trace-mpi` enables MPI-aware result naming

**Setup checkpoint:** the ARE session has **12 cores on one node**; `lulesh2.0`, `mpirun`, and `vtune` are visible to the kernel


<a id="exercise-1"></a>
## Exercise 1 — Establish a baseline (10 min)

**RUN the next cell** as a warm-up and size check. Inspect the complete `run.log` at the printed path. Find the rank count, thread count, actual iterations, final energy/symmetry diagnostics, and LULESH's elapsed time.

Start with `size=30`, `iterations=200`. Aim for several seconds of **application execution**, preferably about 5–20 seconds for this session. If it is too short, increase `lab.size` and/or `lab.iterations` in the configuration cell, then rerun configuration and warm-up. For example, try `size=45`, `iterations=1000`. Keep the final choice fixed for all comparisons.

The helper records **launcher wall time** around `mpirun`. This includes startup and shutdown. LULESH's own timer covers a different region. Label the timer you use and compare the same region in every run.

In [ ]:
warmup = lab.run(threads=1, label="warmup")
print(f"Launcher wall time: {warmup['launcher_wall_seconds']:.3f} s")

Once the workload is suitable, **run three unprofiled repetitions**. The runs use fresh directories, so rerunning a cell does not overwrite a profile.

In [ ]:
baselines = [lab.run(threads=1) for _ in range(3)]
times = [r["launcher_wall_seconds"] for r in baselines]
print("Launcher wall times (s):", [round(t, 3) for t in times])
print(f"Median {statistics.median(times):.3f} s; range {min(times):.3f}–{max(times):.3f} s")

**Exercise 1 checkpoint:** all runs finish successfully; the actual iteration count and numerical diagnostics agree within an appropriate floating-point tolerance for this fixed workload. A zero exit code alone is not a numerical validation. Explain any substantial timing spread before proceeding.

**Record:** mesh size, iteration cap and actual iterations, binary/build, rank/thread counts, binding, timing region, three timings, and a log path in the [worksheet](#worksheet).

<a id="hotspots"></a>
## 3. Hotspots: locate expensive code

All profiling exercises use software sampling (`sampling-mode=sw`). It collects application samples and call stacks without hardware sampling drivers. It still requires the platform's normal process-profiling permissions and adds overhead. Hardware sampling is a separate mode and needs supported counters and access permissions. 

The helper also sets `enable-characterization-insights=false`. VTune enables additional performance insights by default; these collect hardware events even when software sampling is selected. Disabling them avoids that extra hardware requirement and associated PMU setup failures.

For the notebook exercise, the command you run is:

```python
hotspots_1 = lab.run(threads=1, analysis="hotspots")
```

The helper sets the thread count, chooses a fresh results directory, and prints the shell command so you can inspect and reproduce the run. The full command has the following structure:

```bash
export OMP_NUM_THREADS=1 OMP_DYNAMIC=FALSE
export OMP_PLACES=cores OMP_PROC_BIND=close
mpirun -np 1 --map-by slot:PE=1 --bind-to core \
  vtune -collect hotspots -knob sampling-mode=sw \
  -knob enable-characterization-insights=false \
  -trace-mpi -result-dir /path/to/fresh/result \
  -- lulesh2.0 -s 30 -i 200
```

The notebook substitutes your configured workload and the module executable, and creates the fresh result parent automatically. It also adds `--report-bindings` to show which cores Open MPI assigned.

| VTune/application option | Purpose |
|---|---|
| `-collect hotspots` | Find the functions consuming CPU time. |
| `-knob sampling-mode=sw` | Select software sampling. |
| `-knob enable-characterization-insights=false` | Disable the extra hardware-event collection that can cause the `pmu-type` error. |
| `-trace-mpi` | Enable MPI rank identification; particularly useful in Exercise 5. |
| `-result-dir ...` | Choose where to save this collection. The helper supplies a new path each time. |
| `--` | End VTune options and begin the application command. |
| `lulesh2.0 -s 30 -i 200` | Run LULESH with mesh side length 30 per rank and a limit of 200 iterations. |

<details><summary>Reference only: what a hardware Hotspots command looks like</summary>

This example illustrates the other sampling mode. It is **not an exercise** and is not executed by any notebook cell. It requires a system with working hardware sampling; it is not a claim that this collection works in the workshop ARE session.

```bash
export OMP_NUM_THREADS=1 OMP_DYNAMIC=FALSE
export OMP_PLACES=cores OMP_PROC_BIND=close
mpirun -np 1 --map-by slot:PE=1 --bind-to core \
  vtune -collect hotspots -knob sampling-mode=hw \
  -knob enable-stack-collection=false \
  -trace-mpi -result-dir /path/to/fresh/hardware-result \
  -- lulesh2.0 -s 30 -i 200
```

`sampling-mode=hw` selects hardware event sampling. This example disables call-stack collection; that does not remove the hardware-counter requirement. Hardware Hotspots does not automatically provide all the memory metrics of Memory Access analysis. The software command above is the one used throughout the exercises. See [Intel's collection options](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-2/collect.html).

</details>




For mpirun, we have set the following options that control **where the MPI process and its threads can run** within the CPUs already allocated to your ARE session.

| Launcher option | Meaning in this workshop |
|---|---|
| `-np 1` | Start one MPI process, called a rank. |
| `--map-by slot:PE=1` | Use Open MPI's slot-based process placement and assign one processing element (`PE`) per rank. Here each processing element is a CPU core. A slot is a process-placement unit supplied by the allocation; the `1` is the number of cores per rank. |
| `--bind-to core` | Restrict each rank to its assigned core or set of cores using CPU affinity. Open MPI chooses the actual core IDs. |

Together with `OMP_NUM_THREADS=1`, this gives **one MPI rank, one OpenMP thread, bound to one core**. `PE` specifies the number of cores per rank; `OMP_NUM_THREADS` specifies how many OpenMP threads the application should use. The mapping options do not create threads or request another PBS allocation.

For twelve threads, the helper changes both settings together:

| Setting | One thread | Twelve threads |
|---|---|---|
| OpenMP thread count | `OMP_NUM_THREADS=1` | `OMP_NUM_THREADS=12` |
| Cores per MPI rank | `--map-by slot:PE=1` | `--map-by slot:PE=12` |
| Binding policy | `--bind-to core` | `--bind-to core` |

This gives **one MPI rank with twelve OpenMP threads, bound to a set of twelve cores**. `OMP_PLACES=cores` and `OMP_PROC_BIND=close` place the OpenMP threads on cores within that set. The ARE allocation remains twelve cores throughout.

A single rank normally binds to one core. Setting only `OMP_NUM_THREADS=12` does not expand that binding: twelve threads can end up competing for one core. The helper sets `PE=threads` for both baseline and profiled runs, so the twelve-thread run uses `--map-by slot:PE=12 --bind-to core`.

<a id="exercise-2"></a>
## Exercise 2 — Find a hotspot (10 min)

1. Predict whether application calculations or runtime functions will dominate at one thread.
2. **RUN** the collection, summary, and function report cells below in order.
3. Identify the three most expensive functions and their CPU times. Separate LULESH code from MPI/OpenMP/runtime code.
4. Choose one LULESH function to locate and inspect in the matching source in Exercise 3.

A **summary** gives the overall run metrics; a **hotspots report grouped by function** gives the function costs. Inspect any warnings, particularly incomplete collection or missing symbols.

In [ ]:
hotspots_1 = lab.run(threads=1, analysis="hotspots")
print("Actual result directories:")
for result in result_dirs(hotspots_1):
    print(result)

In [ ]:
report(hotspots_1, "summary")

The function report below shows the **ten hottest functions**, sorted by CPU self time, with only the columns needed for this exercise:

| Column | What to look for |
|---|---|
| **Function** | The name of the function to investigate. |
| **CPU Time:Self** | Sampled CPU time attributed to that function itself, excluding time in functions it calls. This is CPU time, not elapsed wall time. |
| **Module** | The executable or library containing the function: for example, `lulesh2.0` versus an MPI or OpenMP library. |

You do **not** need all columns to find the top three functions. The summary above contains the overall spin and overhead times; source locations and other details can be explored in Exercise 3. 

The helper uses VTune's [`-column`](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-2/column.html) option to select `CPU Time:Self`; [`-group-by function`](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-2/group-by.html) retains the function and module. `-limit 10` limits the displayed rows. All collected data remains in the result directory.

If you need **all columns and rows**, run `report(hotspots_1, "hotspots", full=True)`. It reads the existing result without rerunning LULESH. Both views are saved as text files beside the result: `*-hotspots-compact.txt` for the compact view and `*-hotspots.txt` when you request the full view.


In [ ]:
report(hotspots_1, "hotspots")

**Exercise 2 checkpoint:** the report contains your LULESH process and named functions, and you can give the saved result path. Use the exact directory printed above, including any hostname suffix. Do not infer a hotspot ranking from this notebook: it depends on the build, input, and machine.



<a id="metrics"></a>
## 4. Read metrics before drawing conclusions

| Metric | Interpretation | Examples |
|---|---|---|
| Elapsed time | Wall time for the reported collection interval | The collection lasts 6 seconds from start to finish. |
| CPU time | On-CPU time accumulated across threads | Four threads each use a CPU for 2 seconds: 8 CPU-seconds, even if only 2 wall-clock seconds pass. |
| Effective time | CPU time excluding recognised spin/runtime overhead | 5.01 s CPU time minus 0.13 s spin time and 0 s runtime overhead gives 4.88 s effective time. |
| Spin time | Recognised active waiting on the CPU | A thread repeatedly checks a flag while waiting for another thread to finish. |
| Overhead time | Recognised parallel-runtime management | The OpenMP runtime spends CPU time distributing loop iterations among worker threads. |
| Total thread count | Threads observed, including possible helper threads | One main thread and two library helper threads give a total thread count of three. |
| Paused time | Intervals where collection was paused | You pause VTune collection for 1 second and then resume: paused time is 1 second. |

Not all metrics are available for every analysis, CPU, or threading runtime. GCC/OpenMP builds may lack the detailed region/efficiency metrics available with Intel's instrumented runtime. Treat an unavailable metric as unavailable, not zero. 

### A denominator changes the story

Conside the following illustrative example:  four threads each spend 6 seconds on the CPU during an 8-second interval. Total CPU time is 24 seconds; average on-CPU concurrency is `24 / 8 = 3`. That includes spin and hides changes over time.

Now suppose a report shows **6 effectively used physical cores out of 48 detected cores**. That is `6 / 48 = 12.5%` relative to the node. Your ARE session has **12 allocated cores**, so the same average corresponds to `6 / 12 = 50%` of that allocation, provided the quantities use the same physical-core basis. Neither percentage is itself measured parallel efficiency. Check the allocation and metric definition before interpreting a low node-wide percentage.

**Parallel efficiency** compares measured speedup with ideal linear speedup for the number of threads used. The [speedup and parallel efficiency definitions](#speedup-efficiency) before the thread comparison explain each symbol and give a worked example.

<a id="exercise-3"></a>
## Exercise 3 — Interpret and inspect (10 min)

**Goal:** connect a measured hotspot to the code that performs the work, then propose an explanation to test. Use the existing Exercise 2 results and matching source; no new profiling run is needed.


**Inspect one measured hotspot:**

1. Choose a LULESH function from the Exercise 2 Hotspots report and note its **CPU Time:Self**.
2. Set `FUNCTION_NAME` in the next cell to its base C++ name, without arguments or an OpenMP suffix, then run the cell. `CalcForceForNodes` is only an example; use a function from your own report.
3. The output lists matching filenames, line numbers, and source lines. Matches may be declarations or calls, so locate the actual function definition.
4. Open the matching file in the JupyterLab text editor and inspect its loops, array accesses, and calls. If the shared source is outside the file browser, read it from a terminal or copy the file into your own directory for reading.
5. Describe what the function does and write one question to investigate, such as whether its loops become faster with more OpenMP threads.


In [ ]:
FUNCTION_NAME = "CalcForceForNodes"  # EDIT: use a function from your Hotspots report
if LULESH_SOURCE is None or not LULESH_SOURCE.is_dir():
    print("Source inspection is unavailable: ask the instructor for the matching source "
          "directory and set LULESH_SOURCE. Continue interpreting the saved function report.")
else:
    matches = []
    for source in sorted(LULESH_SOURCE.glob("*.cc")):
        for line_number, line in enumerate(source.read_text(errors="replace").splitlines(), 1):
            if FUNCTION_NAME in line:
                matches.append((source, line_number, line.strip()))
    for source, line_number, line in matches[:20]:
        print(f"{source}:{line_number}: {line}")
    if not matches:
        print("No match: use the base C++ function name, without arguments or an OpenMP suffix.")

### Exercise 3 checkpoint

Record your evidence and a question to test:

```text
Function: <chosen LULESH function>
CPU self time: <value from your Hotspots report>
Source: <filename and line number of the function definition>
Observation: <what the function does>
Question to test: <what further measurement would help explain its cost?>
```

Reading source alongside a function report does not establish which exact source line received samples. Keep your observation separate from the explanation you plan to test. If the matching source is unavailable, record that limitation and base your question on the saved report.


<a id="exercise-4"></a>
## Exercise 4 — Use the twelve-core allocation (15 min)

**Predict:** how will runtime change at **1, 2, 4, 6, and 12 threads**? Why could CPU time rise while elapsed time falls?

Keep the executable, mesh, iteration cap, **one MPI rank**, and binding policy fixed. The allocation stays at twelve cores; only the application's thread count changes. The following code cell warms up each new configuration and measures three unprofiled runs. It reuses the one-thread baseline from Exercise 1. Runs are sequential, so their core requests do not add together.

For each thread count, the helper sets both `OMP_NUM_THREADS` and `--map-by slot:PE=<threads>`, keeping `--bind-to core`. For example, twelve threads use `PE=12`, so the rank can use twelve cores.

Check the printed LULESH thread count, numerical diagnostics and actual iterations. If you changed the workload after Exercise 1, rerun the baseline first.

<a id="speedup-efficiency"></a>
### Speedup and parallel efficiency: what do $S_p$ and $E_p$ mean?

Use comparable **unprofiled elapsed times for the same workload**, keeping one MPI rank throughout this exercise.

| Symbol | Definition |
|---|---|
| $p$ | Number of OpenMP threads used in the run: 1, 2, 4, 6, or 12 here. |
| $T_1$ | Median elapsed time with one OpenMP thread. |
| $T_p$ | Median elapsed time with $p$ OpenMP threads. |
| $S_p$ | **Speedup**: how many times faster the $p$-thread run is than the one-thread run. |
| $E_p$ | **Parallel efficiency**: the fraction of ideal linear speedup achieved with $p$ threads. |

$$
S_p = \frac{T_1}{T_p}, \qquad
E_p = \frac{S_p}{p} = \frac{T_1}{pT_p}.
$$

Ideal linear scaling gives $S_p=p$ and $E_p=1$, or **100% efficiency**. For example, ideal scaling on twelve threads would make the same workload run twelve times faster.

Suppose one thread takes $T_1=12$ seconds and twelve threads take $T_{12}=2$ seconds. Then

$$
S_{12}=\frac{12}{2}=6, \qquad
E_{12}=\frac{6}{12}=0.5=50\%.
$$

Twelve threads give a **sixfold speedup**, which is half the ideal linear speedup. Parallel efficiency describes scaling; it is different from CPU utilisation.

The following code uses the **median of three unprofiled launcher elapsed times** at each thread count. It prints $S_p$ in the `speedup` column and $E_p$ as a percentage in the `efficiency` column. Both times include MPI launch and shutdown, so compare the same timing region. Use $p=4$ for the four-thread run and $p=12$ for the twelve-thread run, even though the ARE allocation stays at twelve cores.

In [ ]:
expected = (lab.ranks, lab.size, lab.iterations)
if any((r["ranks"], r["size"], r["iterations"]) != expected for r in baselines):
    raise ValueError("Workload changed: rerun Exercise 1 before comparing")
for threads in THREAD_COUNTS:
    lab.validate(threads)
measurements = {1: baselines}
for threads in THREAD_COUNTS[1:]:
    lab.run(threads=threads, label="warmup")
    measurements[threads] = [lab.run(threads=threads) for _ in range(3)]

print("threads   median launcher s   min s    max s    speedup   efficiency")
t1 = statistics.median(r["launcher_wall_seconds"] for r in measurements[1])
for threads, runs in measurements.items():
    timings = [r["launcher_wall_seconds"] for r in runs]
    median = statistics.median(timings)
    speedup = t1 / median
    print(f"{threads:7d} {median:19.3f} {min(timings):7.3f} {max(timings):8.3f} "
          f"{speedup:10.2f} {speedup / threads:11.1%}")

**RUN a second Hotspots collection at twelve threads**, then compare it with the one-thread result.  Compare absolute function times as well as percentages; a function's percentage can rise because another function got faster.

In [ ]:
hotspots_12 = lab.run(threads=PROFILE_THREADS, analysis="hotspots")
report(hotspots_12, "summary")
report(hotspots_12, "hotspots")

**Exercise 4 checkpoint:** explain whether the repeated unprofiled timings support your prediction. Recheck the numerical diagnostics before claiming an improvement. Use the profiles to suggest a reason.

<details><summary>Possible interpretations — there is no prescribed winner</summary>

A small mesh may have insufficient work per parallel region. Synchronisation, imbalance, and thread management may outweigh the benefit of additional threads. A larger workload can behave differently. Rising CPU time can reflect additional parallel work or waiting. A flat speedup curve alone does not establish a memory-bandwidth bottleneck.

</details>

<a id="next-analysis"></a>
## Reference: choose evidence to test the explanation

Software and hardware analyses answer different questions. The hardware analyses below are **conceptual follow-ups**, outside the notebook exercises.

| Question | Analysis | Evidence to seek |
|---|---|---|
| Where is CPU time spent? | Software `hotspots` — used in this notebook | Functions, call stacks, and recognised runtime waiting |
| How are compute resources used? | `hpc-performance` — reference only | Available CPU, memory and floating-point metrics |
| Is access to data limiting a hot region? | `memory-access` — reference only | Available memory-stall, cache, bandwidth and NUMA metrics |

HPC Performance and Memory Access require supported hardware events and access permissions. Available metrics depend on the processor, VTune version and runtime. We discuss their purpose without collecting them in this notebook.

For memory behaviour, distinguish **bandwidth** (data transferred per second) from **latency** (time waiting for an access). A memory-stalled region can have low bandwidth if it issues too few independent requests. High traffic alone does not prove that reducing traffic will improve elapsed time. Software Hotspots can locate expensive code, but its CPU time alone cannot establish a cache or bandwidth bottleneck.

<a id="exercise-5"></a>
## Exercise 5 — Compare eight MPI ranks with eight OpenMP threads

### First, profile the eight-rank run

Use **eight MPI ranks × one OpenMP thread**, consuming up to eight of the twelve allocated cores. This fits the existing single-node session. **Twelve ranks are invalid for LULESH**.

The helper uses the same **software Hotspots sampling** for the application ranks and passes `-trace-mpi`. Inspect all returned result directories and confirm eight application ranks. Leave `ALLOCATED_CORES=12` and **run the next cell** after the previous experiments finish. It runs an unprofiled baseline, collects software Hotspots, and displays the summary and function reports.

This first step demonstrates **MPI process coverage**. Holding `-s` fixed gives eight times as many elements as the one-rank runs in Exercises 1–4. The comparison below therefore doubles `-s` for the one-rank, eight-thread run to keep the global mesh the same.

In [ ]:
mpi_lab = Lab(binary=lab.binary, results=lab.results,
              allocated_cores=ALLOCATED_CORES, ranks=8,
              size=lab.size, iterations=lab.iterations)
mpi_baseline = mpi_lab.run(threads=1, label="mpi-baseline")
mpi_hotspots = mpi_lab.run(threads=1, analysis="hotspots")
report(mpi_hotspots, "summary")
report(mpi_hotspots, "hotspots")
print("Actual result directories:")
for result in result_dirs(mpi_hotspots):
    print(result)


### Compare the same global mesh on eight cores

**Predict:** will eight MPI ranks or eight OpenMP threads finish sooner, and why?

LULESH's `-s` is the mesh side **per MPI rank**. With the default size used above:

| Configuration | MPI ranks | Threads per rank | `-s` | Global elements | Application cores |
|---|---:|---:|---:|---:|---:|
| MPI | 8 | 1 | 30 | $8 \times 30^3 = 216{,}000$ | 8 |
| OpenMP | 1 | 8 | 60 | $1 \times 60^3 = 216{,}000$ | 8 |

The eight MPI domains form a $2\times2\times2$ layout, so both configurations have a global $60^3$ mesh. If you changed the MPI size, the next cell uses **twice that size** for OpenMP. Both use the same executable and iteration cap (normally `-i 200`); region count, balance and cost options retain the same executable defaults.

The MPI run uses `--map-by slot:PE=1 --bind-to core`; the eight-thread run uses `--map-by slot:PE=8 --bind-to core`. The helper also sets the matching `OMP_NUM_THREADS`. **Eight threads need a rank bound to eight cores.** Keep the 12-core allocation, run sequentially in the same session and inspect the binding output for comparable placement, preferably the same eight cores on one socket.

**RUN the next cell.** It warms up both configurations, then alternates their order over three pairs of **unprofiled** runs. The table uses launcher wall time, including MPI startup and shutdown, consistently for both. Do not reuse Exercise 4 timings: those used the smaller one-rank mesh. Do not use profiled launcher times, which also include VTune collection and finalisation.

Before claiming a faster configuration, check the logs for the expected total elements, rank/thread counts, matching actual iteration counts, and comparable final origin energy and symmetry diagnostics within a justified tolerance. Floating-point reduction order can change the last digits; these printed checks are useful diagnostics, not a full solution comparison.

This compares **two LULESH execution configurations on the same global mesh**. Domain boundaries, memory layout, synchronisation and region assignment can change with the decomposition. 

In [ ]:
openmp_lab = Lab(binary=mpi_lab.binary, results=mpi_lab.results,
                 allocated_cores=ALLOCATED_CORES, ranks=1,
                 size=2 * mpi_lab.size, iterations=mpi_lab.iterations)
comparison_configs = {"MPI": (mpi_lab, 1), "OpenMP": (openmp_lab, 8)}
comparison_runs = {name: [] for name in comparison_configs}

for name, (config, threads) in comparison_configs.items():
    config.validate(threads)
    print(f"{name}: {config.ranks} ranks x {threads} threads, "
          f"-s {config.size}, {config.ranks * config.size**3} total elements, "
          f"iteration cap {config.iterations}")
for name, (config, threads) in comparison_configs.items():
    config.run(threads=threads, label=f"compare-{name.lower()}-warmup")

for repetition in range(3):
    order = list(comparison_configs)
    if repetition % 2:
        order.reverse()
    for name in order:
        config, threads = comparison_configs[name]
        comparison_runs[name].append(
            config.run(threads=threads, label=f"compare-{name.lower()}"))

comparison_medians = {}
print("Configuration   three launcher times (s)       median s    min s    max s")
for name, runs in comparison_runs.items():
    times = [run["launcher_wall_seconds"] for run in runs]
    comparison_medians[name] = statistics.median(times)
    samples = ", ".join(f"{value:.3f}" for value in times)
    print(f"{name:13s} {samples:31s} {comparison_medians[name]:8.3f} "
          f"{min(times):8.3f} {max(times):8.3f}")

mpi_to_openmp_ratio = comparison_medians["MPI"] / comparison_medians["OpenMP"]
print(f"T_MPI / T_OpenMP = {mpi_to_openmp_ratio:.3f}")
print("Above 1: OpenMP has the lower median; below 1: MPI has the lower median.")
print("Check numerical diagnostics, actual iterations and timing variability "
      "before claiming an improvement.")


### Use software profiles to investigate the difference

Collect software Hotspots for the **one-rank, eight-thread, larger-local-mesh** run. Compare its reports with `mpi_hotspots` above, which already covers the same global mesh. Compare absolute function CPU times as well as percentages, MPI routines, and recognised spin/runtime overhead. Both collections use software sampling with additional hardware insights disabled.

Use the unprofiled median ratio $T_{\mathrm{MPI}}/T_{\mathrm{OpenMP}}$ to describe relative runtime. For example, a ratio of 1.25 means the OpenMP configuration is 1.25 times as fast; 0.8 means the MPI configuration is 1.25 times as fast. Both use eight cores, so this ratio is **not** $S_8$ or $E_8$ relative to a one-core baseline. If the timing ranges overlap substantially, collect more repetitions before choosing a winner.

Profiles suggest explanations to test. A large `MPI_Allreduce` cost may include waiting for other ranks; a large OpenMP runtime cost may reflect synchronisation or work distribution. Neither a low spin metric nor high CPU utilisation alone proves good parallel efficiency. Software Hotspots does not measure memory bandwidth or cache misses.


In [ ]:
openmp_hotspots = openmp_lab.run(threads=8, analysis="hotspots",
                                 label="compare-openmp-hotspots")
report(openmp_hotspots, "summary")
report(openmp_hotspots, "hotspots")
print("MPI profile directories:", *result_dirs(mpi_hotspots), sep="\n")
print("OpenMP profile directories:", *result_dirs(openmp_hotspots), sep="\n")


**Exercise 5 checkpoint:**

1. Confirm eight application ranks in the MPI log/result and eight OpenMP threads in the one-rank run. Both configurations should use eight application cores.
2. Verify the same global mesh, matching actual iterations and comparable numerical diagnostics. Record any differences in core placement.
3. Record the three unprofiled timings, median and range for each configuration, plus $T_{\mathrm{MPI}}/T_{\mathrm{OpenMP}}$. Does the difference exceed the observed variability?
4. Compare the two software profiles. Identify one measured function or runtime cost that might help explain the timing difference, and one further measurement that would test that explanation.
5. Explain why a large MPI cost can include waiting for work on another rank, and why changing decomposition can change work as well as communication.


<a id="worksheet"></a>
## Record your measurements

**EDIT this Markdown cell** after each exercise. Mark missing metrics `not collected` or `not available`.

ARE session: **12 CPU cores, one node**. Job ID / hostname: …

Loaded `lulesh` module / executable path / binary hash: …

Matching source / build details supplied by the instructor: …  MPI / VTune versions: …

Mesh side per rank: …  Iteration cap / actual iterations: …  Numerical checks / tolerance: …

| Run | Ranks × threads | Timer / 3 unprofiled times / median | Dominant function and CPU time | Log / result path |
|---|---|---|---|---|
| Baseline | 1 × 1 | | | |
| Thread comparison | 1 × 2 | | | |
| Thread comparison | 1 × 4 | | | |
| Thread comparison | 1 × 6 | | | |
| Thread comparison | 1 × 12 | | | |
| Hotspots | 1 × 1 | Profile; separate from baseline | | |
| Hotspots | 1 × 12 | Profile; separate from baseline | | |
| Exercise 5: MPI, local `-s 30` by default | 8 × 1 | | | |
| Exercise 5: OpenMP, local `-s 60` by default | 1 × 8 | | | |

**Does using all twelve cores improve this workload? Evidence and variability:** …

**Exercise 5: global elements / actual iterations / numerical checks:** …

**MPI versus OpenMP median ratio and timing variability:** …

**What the two profiles suggest, and what remains uncertain:** …

**My hypothesis and the next change I would test:** …

**Success criterion (correctness and elapsed time):** …

## Troubleshooting

| Symptom | Next check |
|---|---|
| Session reports a different core count | Check the ARE session card; relaunch with 12 cores on one node |
| Storage or repository path is missing | Check Storage in the ARE launch form and your own project/user directory |
| `lulesh2.0` missing | Check `lulesh` in ARE Modules, the private Module directories setting, and mounted storage; relaunch |
| LULESH reports a missing shared library | Ask the instructor to check the private module runtime dependencies |
| `LULESH_SOURCE` unset or unreadable | Obtain the matching source path; function-level profiling can continue |
| `vtune` missing inside the notebook | Correct Modules in the ARE launch form and relaunch the session |
| MPI binding error or wrong thread count | Open MPI version, allocated cores, `--map-by`, `OMP_NUM_THREADS` |
| “Num processors must be a cube” | Use 1 or 8 ranks; the 12-core run is 1 rank × 12 threads |
| No result at the requested name | Check the actual hostname-suffixed result path and collection log |
| Hotspots fails with an internal `pmu-type` error | Check that `lab.py` adds `-knob enable-characterization-insights=false`; after updating the helper, restart the kernel and rerun setup |
| Unknown functions or no source | Ask the instructor to verify the prebuilt binary symbols and matching source path; inspect finalisation warnings |
| Few samples / unstable rankings | Increase application work, then rerun baseline and profile at the same size |
| Low whole-node utilisation | Compare detected node cores with this session's 12-core budget |
| GUI cannot open | Use reports/source in JupyterLab; GUI needs a separate display-enabled environment |

## Final concept checks

1. Why use optimisation and debug symbols together?
2. How can CPU time exceed elapsed time?
3. What remains unknown after finding the most expensive function?
4. Why do the helper's profiled launcher time and VTune's elapsed time differ?
5. Why does keeping `-s` fixed while increasing ranks change the total work?
6. What evidence would persuade you that your next change improved the application?
7. Why can this 12-core session run 1 rank × 12 threads, but not 12 LULESH ranks?
8. What triggers software and hardware samples, and which mode do these exercises use?

<details><summary>Reference answers</summary>

1. Preserve realistic execution while retaining source attribution.
2. CPU time sums activity across threads.
3. Its cause, scope for improvement, and effect on end-to-end runtime.
4. The launcher measurement includes startup, collection, and result finalisation; the profiler reports its collection interval.
5. `-s` describes a local cube, so total elements are `ranks × s³`.
6. Correct numerical behaviour and a repeatable improvement for the same workload and timing region, with the environment and changes recorded.
7. The resource budget is ranks × threads, while LULESH separately requires a cubic MPI rank count. One rank is valid; twelve ranks are not.
8. Software sampling uses periodic timer samples; hardware sampling uses selected processor events reaching a threshold. All profiling exercises use software sampling with additional hardware insights disabled.

</details>